# 🎙️ Sesame Clone — a conversational voice assistant on a free Colab GPU

**Whisper (ears) → Gemma 3 (brain) → Sesame CSM-1B (voice)**

This notebook reproduces the *architecture* behind Sesame's demo — the thing that makes it
sound natural — not just "text in, audio out". Everything runs on a free Colab T4.

---

## What Sesame's research actually says

From [*Crossing the uncanny valley of conversational voice*](https://www.sesame.com/research/crossing_the_uncanny_valley_of_voice)
(Schalkwyk, Kumar, Lyth, Eskimez, Hodari, Resnick, Sanabria, Jiang et al.):

**The problem they name is "one-to-many".** One sentence can be spoken in a thousand valid ways.
Text alone doesn't tell you which one. Classic TTS has no mechanism to choose, so it hedges and
picks the safe, flat, neutral reading every time. That flatness *is* the uncanny valley.

**Their fix: condition on the conversation, including its audio.** CSM (Conversational Speech
Model) is a single-stage multimodal transformer that takes interleaved **text *and* audio** history
as context and generates RVQ audio codes directly. Prosody stops being guessed from a lone
sentence and starts being inferred from how the conversation has actually been going.

**Architecture — two autoregressive transformers over a split-RVQ tokenizer:**

| Component | Job |
|---|---|
| **Backbone** (Llama-style, 1B in CSM-1B) | Reads interleaved text + audio, predicts **codebook 0** — the *semantic* token |
| **Depth / audio decoder** (~100M, 4 layers) | Predicts the remaining **31 acoustic codebooks** for that frame |
| **Mimi codec** (Kyutai) | 24 kHz audio ⇄ discrete tokens at **12.5 Hz** (one frame per 80 ms) |

Splitting it this way keeps the big model on meaning and a tiny model on timbre — that's what
makes it fast enough to hold a conversation.

**Compute amortization (their training trick).** The audio decoder would need an effective batch of
`B × S × N` (batch × sequence × 32 codebooks) — brutal memory. So they train the decoder on a
random **1/16 subset of frames** while codebook 0 trains on *every* frame. No perceptible quality
loss. This notebook exposes that exact knob in the fine-tuning section
(`depth_decoder_labels_ratio=1/16`).

**Scale.** Trained on ~**1M hours** of mostly-English audio, in three sizes: Tiny (1B backbone /
100M decoder — the open one), Small (3B / 250M), Medium (8B / 300M). Bigger = more contextually
appropriate, not just cleaner.

**How they measured it.** Word-error-rate and speaker similarity were already saturated, so they
invented two context-sensitive tests: **homograph disambiguation** (does it say "lead" the metal or
"lead" the verb?) and **pronunciation continuation consistency** (does a name keep the same
pronunciation across turns?). Subjectively, with CMOS studies on Expresso: on *isolated* sentences
CSM is near parity with human recordings — but **once conversational context is included, humans
still clearly prefer real humans.** They published the gap rather than hiding it.

**What they said is still missing:** full duplex — real turn-taking, interruption, and pacing.
CSM generates speech; it doesn't yet decide *when* to speak.

---

## What that means for the code below

Four design rules, taken straight from the paper. These are the difference between this and any
generic TTS demo:

1. **The context buffer stores audio, not just text.** Every turn — yours *and* the assistant's —
   is kept as `(speaker_id, text, waveform)` and fed back into CSM. This is the whole ballgame.
2. **A pinned voice anchor.** CSM-1B is a *base* model with no fixed voice; with an empty context it
   invents a new speaker every call. We bootstrap one reference utterance and pin it to the front of
   the context forever.
3. **A real frame budget.** The backbone tops out at 2048 positions and audio burns 12.5 of them per
   second. We count text tokens and audio frames exactly, and evict oldest-first while never
   evicting the anchor.
4. **Sentence-level chunking.** Speak sentence by sentence, feeding each finished chunk back as
   context. First audio arrives sooner and prosody still carries across the sentence boundary.

> ⚠️ **Responsible use.** Sesame's license explicitly prohibits impersonating real people without
> consent, deception, and fraud. Note also that the original Sesame repo watermarks its output with
> `silentcipher`; the 🤗 Transformers integration used here does **not**. If you ship anything from
> this, add watermarking back. Details in the last cell.

## Step 0 — Get a GPU

`Runtime → Change runtime type → T4 GPU`, then run this.

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "!! No GPU. Runtime > Change runtime type > T4 GPU")

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > T4 GPU"

MAJOR, MINOR = torch.cuda.get_device_capability()
SUPPORTS_BF16 = MAJOR >= 8
print(f"torch {torch.__version__} | compute capability {MAJOR}.{MINOR} | bf16 supported: {SUPPORTS_BF16}")
if not SUPPORTS_BF16:
    print("\nT4 detected (sm_75): no native bfloat16. CSM will run in float32 (~5 GB, safe).")
    print("fp16 is ~4x faster but CSM's RVQ sampling can go unstable in it -- there's a knob below.")

## Step 1 — Install

CSM needs `transformers >= 4.52.1`. **If pip upgrades anything, restart the runtime**
(`Runtime → Restart session`) and then continue from the *next* cell — you don't need to re-run this one.

In [ ]:
%pip install -q -U "transformers>=4.56" accelerate bitsandbytes peft soundfile librosa datasets

import transformers
print("transformers", transformers.__version__)
print("\nIf that upgraded packages: Runtime > Restart session, then continue from the next cell.")

## Step 2 — Hugging Face auth

**Both models are gated.** Open each link, click "Agree and access repository" (instant), *then*
log in here. Skipping this is the #1 reason this notebook fails with a `401`/`403`.

- 🔗 [`sesame/csm-1b`](https://huggingface.co/sesame/csm-1b)
- 🔗 [`google/gemma-3-4b-it`](https://huggingface.co/google/gemma-3-4b-it) (and/or [`gemma-3-1b-it`](https://huggingface.co/google/gemma-3-1b-it))

Best practice: put your token in the Colab **secrets** panel (🔑 in the left sidebar) as `HF_TOKEN`
with notebook access enabled — then this cell is fully automatic.

In [ ]:
import os
from huggingface_hub import login, whoami

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if token:
    login(token=token, add_to_git_credential=False)
else:
    login()  # interactive prompt

print("Logged in as:", whoami()["name"])

## Step 3 — Configuration

Memory budget on a 15 GB T4, with the defaults below:

| Model | Precision | VRAM |
|---|---|---|
| CSM-1B + Mimi | fp32 | ~5.6 GB |
| Gemma-3-4b-it | 4-bit NF4 | ~3.2 GB |
| Whisper-small | fp16 | ~0.5 GB |
| KV cache + activations | | ~2 GB |
| **Total** | | **~11 GB** ✅ |

Tight but fine. If you OOM, set `LLM_MODEL_ID = "google/gemma-3-1b-it"`.

In [ ]:
import torch

# ---- models -----------------------------------------------------------------
CSM_MODEL_ID = "sesame/csm-1b"
LLM_MODEL_ID = "google/gemma-3-4b-it"     # low-memory fallback: "google/gemma-3-1b-it"
ASR_MODEL_ID = "openai/whisper-small"     # "openai/whisper-base" is faster, sloppier
LOAD_LLM_IN_4BIT = True                   # set False only if you picked the 1b model

# ---- precision --------------------------------------------------------------
# T4 (sm_75) has no bfloat16. float32 is the safe default; fp16 is ~4x faster but
# CSM's RVQ sampling can produce static/garbled audio in it. Try it, keep it if it sounds fine.
CSM_USE_FP16_ON_T4 = False

if SUPPORTS_BF16:
    CSM_DTYPE = torch.bfloat16
else:
    CSM_DTYPE = torch.float16 if CSM_USE_FP16_ON_T4 else torch.float32
LLM_COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
DEVICE = "cuda"

# ---- CSM / Mimi constants (from the paper + config) -------------------------
SAMPLE_RATE   = 24_000    # Mimi operates at 24 kHz
FRAME_RATE    = 12.5      # 12.5 Hz -> one audio frame per 80 ms
NUM_CODEBOOKS = 32        # 1 semantic (backbone) + 31 acoustic (depth decoder)
CSM_MAX_SEQ   = 2048      # backbone max_position_embeddings

# ---- speaker ids ------------------------------------------------------------
ASSISTANT = 0
USER      = 1

# ---- transformers 4.x/5.x compatibility -------------------------------------
import transformers
DTYPE_KEY = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"

print(f"CSM dtype: {CSM_DTYPE} | LLM: {LLM_MODEL_ID} (4-bit={LOAD_LLM_IN_4BIT}) | kwarg: {DTYPE_KEY}=")

## Step 4 — Load the three models

First run downloads ~10 GB, so give it a few minutes.

In [ ]:
import gc, time, numpy as np, torch
from transformers import (AutoProcessor, CsmForConditionalGeneration, AutoTokenizer,
                          AutoModelForCausalLM, BitsAndBytesConfig, pipeline)

def vram(tag=""):
    print(f"   VRAM {torch.cuda.memory_allocated()/1e9:5.2f} GB allocated  {tag}")

# ---------------------------------------------------------------- CSM (voice)
print("Loading CSM-1B ...")
t0 = time.time()
csm_processor = AutoProcessor.from_pretrained(CSM_MODEL_ID)
csm_model = CsmForConditionalGeneration.from_pretrained(
    CSM_MODEL_ID, **{DTYPE_KEY: CSM_DTYPE}
).to(DEVICE).eval()
print(f"   done in {time.time()-t0:.0f}s"); vram("after CSM")

# ---------------------------------------------------------------- Gemma 3 (brain)
print(f"Loading {LLM_MODEL_ID} ...")
t0 = time.time()
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)

llm_kwargs = {"device_map": "auto", DTYPE_KEY: LLM_COMPUTE_DTYPE}
if LOAD_LLM_IN_4BIT:
    llm_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=LLM_COMPUTE_DTYPE,
    )

try:
    llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID, **llm_kwargs)
except (ValueError, KeyError):
    # gemma-3-4b+ are multimodal checkpoints (Gemma3ForConditionalGeneration)
    from transformers import AutoModelForImageTextToText
    llm_model = AutoModelForImageTextToText.from_pretrained(LLM_MODEL_ID, **llm_kwargs)
llm_model.eval()
print(f"   done in {time.time()-t0:.0f}s"); vram("after Gemma")

# ---------------------------------------------------------------- Whisper (ears)
print(f"Loading {ASR_MODEL_ID} ...")
t0 = time.time()
asr = pipeline("automatic-speech-recognition", model=ASR_MODEL_ID,
               device=0, **{DTYPE_KEY: torch.float16})
print(f"   done in {time.time()-t0:.0f}s"); vram("after Whisper")

gc.collect(); torch.cuda.empty_cache()
print("\nAll three models are up.")

## Step 5 — Audio plumbing

Colab can't touch your microphone from Python, so we go through a bit of JavaScript
(`MediaRecorder`) and convert the resulting WebM/Opus to 24 kHz mono with `ffmpeg`.

In [ ]:
import base64, io, re, subprocess, tempfile, os
import numpy as np, soundfile as sf, librosa
from IPython.display import Audio, Javascript, display
from pathlib import Path


def play(wav, sr=SAMPLE_RATE, autoplay=True):
    """Render an audio player for a float32 numpy waveform."""
    display(Audio(np.asarray(wav, dtype=np.float32), rate=sr, autoplay=autoplay))


def save_wav(wav, path, sr=SAMPLE_RATE):
    sf.write(path, np.asarray(wav, dtype=np.float32), sr)
    return path


def load_wav(path, sr=SAMPLE_RATE):
    """Load anything ffmpeg/libsndfile understands as mono float32 at `sr`."""
    wav, _ = librosa.load(path, sr=sr, mono=True)
    return wav.astype(np.float32)


def to_16k(wav, sr=SAMPLE_RATE):
    """Whisper wants 16 kHz; CSM/Mimi want 24 kHz. Keep both around."""
    return librosa.resample(np.asarray(wav, dtype=np.float32), orig_sr=sr, target_sr=16_000)


def secs(wav, sr=SAMPLE_RATE):
    return len(wav) / sr


# --------------------------------------------------------------- microphone
_RECORD_JS = """
const sleep = t => new Promise(r => setTimeout(r, t));
var record = time => new Promise(async resolve => {
  const stream   = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks   = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.onstop = async () => {
    const blob   = new Blob(chunks);
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  };
  recorder.start();
  await sleep(time);
  recorder.stop();
  stream.getTracks().forEach(t => t.stop());
});
"""


def record_mic(seconds=5):
    """Record `seconds` of microphone audio in the browser -> 24 kHz mono float32.

    Colab will ask for mic permission the first time. Returns None if it fails
    (headless browser, denied permission, etc.) so callers can fall back to typing.
    """
    from google.colab import output
    print(f"🔴 Recording {seconds}s ... speak now")
    display(Javascript(_RECORD_JS))
    try:
        data_url = output.eval_js(f"record({int(seconds * 1000)})")
    except Exception as e:
        print("Microphone unavailable:", e)
        return None
    print("⏹️  Done, transcribing ...")

    raw = base64.b64decode(data_url.split(",")[1])
    with tempfile.TemporaryDirectory() as d:
        src, dst = Path(d) / "in.webm", Path(d) / "out.wav"
        src.write_bytes(raw)
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
             "-ar", str(SAMPLE_RATE), "-ac", "1", str(dst)],
            check=True,
        )
        return load_wav(dst)


def transcribe(wav, sr=SAMPLE_RATE):
    out = asr(to_16k(wav, sr), generate_kwargs={"language": "en", "task": "transcribe"})
    return out["text"].strip()


print("Audio helpers ready.")

## Step 6 — The conversational context engine

**This is the part that matters.** Everything else in this notebook is off-the-shelf.

CSM's input is an interleaved sequence of `[speaker]text` tokens and Mimi audio frames. A turn that
carries its **audio** teaches the model how this conversation *sounds*; a turn carrying only text
teaches it only what was said. So we keep the waveforms.

Two things this class has to get right:

**Frame budgeting.** The backbone has 2048 positions. Audio spends 12.5 of them per second, so 60
seconds of history is 750 positions before a single word of text. We count text tokens exactly (via
the processor's tokenizer) and audio frames exactly (`ceil(samples / 24000 × 12.5) + 1` for the EOS
frame), then evict oldest-first until we fit — reserving room for what we're about to generate.

**Anchor pinning.** The voice anchor is *never* evicted. Drop it and the assistant's timbre resets
to a random stranger mid-conversation.

In [ ]:
import math
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class Turn:
    """One utterance. `audio` is 24 kHz mono float32 -- keeping it is the whole point."""
    speaker: int
    text: str
    audio: Optional[np.ndarray] = None

    def cost(self) -> int:
        """Positions this turn consumes in the backbone's 2048-token window."""
        # text: "[spk]text" plus BOS/EOS framing
        n_text = len(csm_processor.tokenizer.encode(f"[{self.speaker}]{self.text}")) + 2
        n_audio = 0
        if self.audio is not None and len(self.audio) > 0:
            n_audio = math.ceil(len(self.audio) / SAMPLE_RATE * FRAME_RATE) + 1  # +1 = EOS frame
        return n_text + n_audio

    def to_message(self, with_audio: bool = True) -> dict:
        content = [{"type": "text", "text": self.text}]
        if with_audio and self.audio is not None and len(self.audio) > 0:
            # the processor accepts a raw numpy array under the "path" key
            content.append({"type": "audio", "path": np.asarray(self.audio, dtype=np.float32)})
        return {"role": str(self.speaker), "content": content}


class Conversation:
    """Rolling text+audio history for CSM, with a pinned voice anchor.

    Sesame's core claim: prosody should be inferred from conversational context, not guessed
    from an isolated sentence. That only works if the context still has audio in it.
    """

    def __init__(self, anchors: Optional[List[Turn]] = None, max_seq: int = CSM_MAX_SEQ):
        self.anchors: List[Turn] = list(anchors or [])   # never evicted
        self.turns: List[Turn] = []                      # evicted oldest-first
        self.max_seq = max_seq

    def add(self, turn: Turn):
        self.turns.append(turn)

    def add_anchor(self, turn: Turn):
        self.anchors.append(turn)

    def build(self, next_text: str, next_speaker: int, reserve_frames: int = 300):
        """Return the message list for `apply_chat_template`, trimmed to fit.

        `reserve_frames` is headroom for the audio we're about to generate
        (12.5 frames per second of speech).
        """
        pending = Turn(next_speaker, next_text, None)
        budget = self.max_seq - reserve_frames - pending.cost() - 16  # 16 = safety margin
        budget -= sum(t.cost() for t in self.anchors)

        if budget < 0:
            raise ValueError(
                "Voice anchors alone overflow the context window -- use shorter anchor clips."
            )

        kept, used = [], 0
        for turn in reversed(self.turns):          # newest first
            c = turn.cost()
            if used + c > budget:
                # try the same turn text-only before giving up on it entirely
                text_only = Turn(turn.speaker, turn.text, None)
                if used + text_only.cost() <= budget:
                    kept.append(text_only)
                    used += text_only.cost()
                continue
            kept.append(turn)
            used += c
        kept.reverse()

        msgs = [t.to_message() for t in self.anchors]
        msgs += [t.to_message() for t in kept]
        msgs.append({"role": str(next_speaker), "content": [{"type": "text", "text": next_text}]})
        return msgs

    def transcript(self, n: int = 6):
        for t in self.turns[-n:]:
            who = "assistant" if t.speaker == ASSISTANT else "user"
            dur = f" [{secs(t.audio):.1f}s audio]" if t.audio is not None else " [text only]"
            print(f"  {who:>9}: {t.text}{dur}")


print("Conversation engine ready.")

## Step 7 — Synthesis, with sentence chunking

Two practical notes:

**Chunking.** We split the reply into sentences and synthesize them one at a time, appending each
finished chunk to the context before the next. First audio arrives in ~1–2 s instead of after the
whole paragraph, and because each chunk sees the previous one, prosody carries across the boundary
instead of restarting flat. It also keeps individual generations short, which is where CSM is most
stable.

**Text cleaning.** CSM will cheerfully read `**bold**` as "asterisk asterisk bold". Markdown, emoji
and bullets get stripped before they reach the model.

In [ ]:
import re

_EMOJI = re.compile(
    "[" "\U0001F300-\U0001FAFF" "\U00002600-\U000027BF" "\U0001F1E6-\U0001F1FF" "←-⇿"
    "⬀-⯿" "️" "]+", flags=re.UNICODE
)


def clean_for_speech(text: str) -> str:
    """Strip anything CSM would try to pronounce literally."""
    text = _EMOJI.sub("", text)
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)      # code fences
    text = re.sub(r"[*_`#>|]+", " ", text)                        # markdown syntax
    text = re.sub(r"^\s*[-•+]\s+", "", text, flags=re.MULTILINE)  # bullets
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)               # links -> label
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)                  # no space before punctuation
    return text.strip()


def split_sentences(text: str, max_chars: int = 180, min_chars: int = 25) -> List[str]:
    """Sentence-ish chunks sized for low-latency synthesis."""
    parts = re.split(r"(?<=[.!?…])\s+", text)

    # long sentences: break on clause boundaries rather than truncating
    expanded = []
    for p in parts:
        while len(p) > max_chars:
            cut = max(p.rfind(", ", 0, max_chars), p.rfind("; ", 0, max_chars),
                      p.rfind(" and ", 0, max_chars))
            if cut < min_chars:
                cut = p.rfind(" ", 0, max_chars)
            if cut < min_chars:
                break
            expanded.append(p[:cut + 1].strip())
            p = p[cut + 1:].strip()
        if p:
            expanded.append(p)

    # merge runts ("Sure." "Okay.") into their neighbour so they get real prosody
    merged = []
    for p in expanded:
        if merged and len(p) < min_chars and len(merged[-1]) + len(p) < max_chars:
            merged[-1] = f"{merged[-1]} {p}"
        else:
            merged.append(p)
    return [m for m in merged if m.strip()]


@torch.inference_mode()
def csm_generate(messages, max_seconds: float = 15.0, temperature: float = 0.9,
                 top_k: int = 50) -> np.ndarray:
    """One CSM forward pass: messages -> waveform.

    Backbone samples codebook 0 (semantic); the depth decoder samples codebooks 1-31
    (acoustic). Both get their own sampling params -- `depth_decoder_*` kwargs.
    """
    inputs = csm_processor.apply_chat_template(messages, tokenize=True, return_dict=True)
    inputs = inputs.to(DEVICE)
    if inputs.get("input_values") is not None:
        inputs["input_values"] = inputs["input_values"].to(CSM_DTYPE)

    audio = csm_model.generate(
        **inputs,
        output_audio=True,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        max_new_tokens=int(max_seconds * FRAME_RATE),
        depth_decoder_do_sample=True,
        depth_decoder_temperature=temperature,
        depth_decoder_top_k=top_k,
    )
    return audio[0].float().cpu().numpy()


def speak(text: str, conv: Conversation, speaker: int = ASSISTANT,
          temperature: float = 0.9, verbose: bool = True):
    """Synthesize `text` in conversational context, chunk by chunk.

    Returns (full_waveform, [Turn, ...]) -- the turns are already context-shaped, so the
    caller can append them and the next reply will hear this one.
    """
    chunks = split_sentences(clean_for_speech(text))
    pieces, new_turns = [], []
    t_start = time.time()

    for i, chunk in enumerate(chunks):
        # earlier chunks of *this* reply are context for the next one
        scratch = Conversation(conv.anchors, conv.max_seq)
        scratch.turns = conv.turns + new_turns

        est = min(15.0, max(3.0, len(chunk) / 13.0))  # ~13 chars/sec of speech
        msgs = scratch.build(chunk, speaker, reserve_frames=int(est * FRAME_RATE) + 40)

        wav = csm_generate(msgs, max_seconds=est + 3.0, temperature=temperature)
        pieces.append(wav)
        new_turns.append(Turn(speaker, chunk, wav))

        if verbose and i == 0:
            print(f"   ⏱️  first audio in {time.time()-t_start:.1f}s")

    full = np.concatenate(pieces) if pieces else np.zeros(0, dtype=np.float32)
    if verbose and len(full):
        gen = time.time() - t_start
        print(f"   ⏱️  {secs(full):.1f}s of audio in {gen:.1f}s "
              f"(RTF {gen/max(secs(full), 1e-6):.2f}x, {len(chunks)} chunk(s))")
    return full, new_turns


print("Synthesis ready.")

## Step 8 — Bootstrap a voice anchor

CSM-1B ships as a **base** model. It was never fine-tuned on a specific voice, and with an empty
context it samples a fresh speaker identity every single call — your assistant would sound like a
different person in every sentence. (Sesame's actual demo runs a *fine-tuned* variant. That's the
one real difference between this notebook and their product.)

The fix without any training: generate one utterance, decide you like it, and pin it as context
forever. Every later generation is conditioned on it and inherits the timbre.

Run the cell, listen, and **re-run with a different `VOICE_SEED` until you find a voice you like.**
This is the single highest-leverage knob in the notebook.

In [ ]:
VOICE_SEED = 42                # <-- change this and re-run until you like the voice
ANCHOR_TEXT = ("Hey, good to hear from you. I'm doing great, thanks for asking. "
               "So what did you want to talk about today?")

torch.manual_seed(VOICE_SEED)
np.random.seed(VOICE_SEED)

print(f"Bootstrapping voice with seed {VOICE_SEED} ...")
_boot = Conversation()
anchor_wav = csm_generate(
    [{"role": str(ASSISTANT), "content": [{"type": "text", "text": ANCHOR_TEXT}]}],
    max_seconds=12.0, temperature=0.85,
)

VOICE_ANCHOR = Turn(ASSISTANT, ANCHOR_TEXT, anchor_wav)
print(f"Anchor: {secs(anchor_wav):.1f}s, {VOICE_ANCHOR.cost()} context positions")
save_wav(anchor_wav, "voice_anchor.wav")
play(anchor_wav)

### Optional — use *your own* voice as the anchor

Record ~10 seconds instead of bootstrapping. This is voice cloning, so: **only record yourself, or
someone who has actually agreed to it.** Sesame's license prohibits the rest, and so does common
decency.

Skip this cell to keep the bootstrapped voice.

In [ ]:
USE_MY_VOICE = False   # flip to True, then run

if USE_MY_VOICE:
    my_wav = record_mic(seconds=10)
    if my_wav is not None:
        my_text = transcribe(my_wav)
        print("Transcript:", my_text)
        VOICE_ANCHOR = Turn(ASSISTANT, my_text, my_wav)
        print(f"Anchor replaced: {secs(my_wav):.1f}s, {VOICE_ANCHOR.cost()} positions")
        play(my_wav, autoplay=False)
else:
    print("Keeping the bootstrapped anchor. Set USE_MY_VOICE = True to override.")

## Step 9 — Hear the thesis: context vs. no context

Sesame's central claim in one experiment. Same sentence, twice:

- **A** — no context. The model has nothing to go on and gives you the safe, flat, average reading.
- **B** — with an excited, fast-paced conversation in the context window. Same words, different
  delivery, because the model can now infer *how* this conversation is going.

If B doesn't sound noticeably more alive than A, your anchor is weak — go back and reroll the seed.

In [ ]:
TEST_LINE = "Wait, are you serious? That is completely unbelievable."

print("── A: no context ─────────────────────────────────────────")
wav_a = csm_generate(
    [{"role": str(ASSISTANT), "content": [{"type": "text", "text": TEST_LINE}]}],
    max_seconds=8.0,
)
play(wav_a, autoplay=False)

print("\n── B: same line, inside an excited conversation ──────────")
ctx = Conversation(anchors=[VOICE_ANCHOR])

lead_in = [
    (USER,      "Okay I have to tell you something, you are not going to believe this."),
    (ASSISTANT, "Oh no. What happened?"),
    (USER,      "I just found out we got the grant. The whole thing. Every penny we asked for!"),
]
for spk, line in lead_in:
    w, turns = speak(line, ctx, speaker=spk, verbose=False)
    ctx.turns.extend(turns)

msgs = ctx.build(TEST_LINE, ASSISTANT, reserve_frames=120)
wav_b = csm_generate(msgs, max_seconds=8.0)
play(wav_b, autoplay=False)

print("\nSame text, same weights. The only difference is what was in the context window.")

## Step 10 — Gemma 3 as the conversational brain

CSM generates speech; it cannot generate text (Sesame is explicit about this in their FAQ — it's an
audio model, not a general multimodal LLM). So Gemma 3 writes the words.

The system prompt matters more than you'd expect. An LLM tuned for chat writes for the *eye*:
bullet points, headers, bold, em-dashes, 200-word answers. All of that sounds terrible out loud. We
force it to write for the ear — short, spoken, contraction-heavy, no formatting.

In [ ]:
SYSTEM_PROMPT = """You are a warm, quick-witted voice companion having a real spoken conversation.

Rules, because every word you write will be spoken aloud:
- Keep replies to 1-3 short sentences. This is talking, not writing.
- Use contractions and everyday words. Sound like a person, not a manual.
- Never use markdown, bullet points, headers, emoji, or parentheses.
- Never use numbered lists. If you must enumerate, say "first... then..." out loud.
- Write numbers, dates and units the way you'd say them: "about twenty bucks", not "$20".
- React to what they actually said. Ask a real follow-up question when it fits.
- If you don't know something, just say so briefly."""


@torch.inference_mode()
def llm_reply(history: List[dict], max_new_tokens: int = 110, temperature: float = 0.8) -> str:
    """history: [{"role": "user"/"assistant", "content": str}, ...]"""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history

    inputs = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(llm_model.device)

    out = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        pad_token_id=llm_tokenizer.eos_token_id,
    )
    reply = llm_tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return clean_for_speech(reply)


# smoke test
print("Gemma:", llm_reply([{"role": "user", "content": "Hey, what have you been up to?"}]))

## Step 11 — Wire it together

The full loop. Note what gets stored where:

- **Gemma's history** is text only — that's all an LLM can use.
- **CSM's context keeps the waveforms**, including *your recorded voice* for user turns. The model
  hears the actual conversation, not a transcript of it. That's rule #1 from the paper, and it's why
  the speaker-id tokens matter: `[1]` marks your turns so CSM knows whose voice is whose and doesn't
  start answering in your voice.

In [ ]:
class VoiceAgent:
    """Whisper -> Gemma 3 -> CSM, sharing one conversation that remembers audio."""

    def __init__(self, anchor: Turn, temperature_llm=0.8, temperature_tts=0.9):
        self.ctx = Conversation(anchors=[anchor])   # CSM: text + audio
        self.history: List[dict] = []               # Gemma: text only
        self.t_llm = temperature_llm
        self.t_tts = temperature_tts

    def turn(self, user_text: str, user_audio: Optional[np.ndarray] = None,
             autoplay: bool = True):
        # 1. remember what the user said -- with their real audio if we have it
        self.ctx.add(Turn(USER, user_text, user_audio))
        self.history.append({"role": "user", "content": user_text})

        # 2. think
        t0 = time.time()
        reply = llm_reply(self.history, temperature=self.t_llm)
        print(f"🤖 {reply}")
        print(f"   ⏱️  Gemma: {time.time()-t0:.1f}s")

        # 3. speak, conditioned on everything above
        wav, new_turns = speak(reply, self.ctx, ASSISTANT, temperature=self.t_tts)

        # 4. remember our own voice too -- next reply hears this one
        self.ctx.turns.extend(new_turns)
        self.history.append({"role": "assistant", "content": reply})

        if len(wav):
            play(wav, autoplay=autoplay)
        return reply, wav

    def say_text(self, user_text: str, **kw):
        """Typed input -- no user audio in context."""
        return self.turn(user_text, None, **kw)

    def say_voice(self, seconds: int = 5, **kw):
        """Spoken input -- the user's waveform goes into CSM's context."""
        wav = record_mic(seconds)
        if wav is None:
            print("No audio captured.")
            return None, None
        text = transcribe(wav)
        if not text:
            print("Didn't catch that.")
            return None, None
        print(f"🗣️  You: {text}")
        return self.turn(text, wav, **kw)

    def reset(self):
        self.ctx.turns.clear()
        self.history.clear()
        print("Conversation cleared (voice anchor kept).")


agent = VoiceAgent(VOICE_ANCHOR)
print("Agent ready.")

## Step 12 — Talk to it 🎤

Run the cell, allow microphone access, speak when it says *Recording*. Re-run for each turn — the
context (and the voice) persists across cells.

**If the mic doesn't work** in your browser, skip to Step 13 and type instead.

In [ ]:
RECORD_SECONDS = 6

agent.say_voice(seconds=RECORD_SECONDS)

## Step 13 — Or just type

Same pipeline minus Whisper. Useful for iterating fast, and for checking the assistant's voice stays
put across many turns.

In [ ]:
agent.say_text("Hi! Can you tell me what makes your voice sound different from normal text to speech?")

In [ ]:
# keep going -- watch the voice stay consistent and the prosody react to the conversation
agent.say_text("That's interesting. Does it get better the longer we talk?")

In [ ]:
print("Context window usage:")
used = sum(t.cost() for t in agent.ctx.anchors) + sum(t.cost() for t in agent.ctx.turns)
print(f"  {used} / {CSM_MAX_SEQ} positions "
      f"({sum(secs(t.audio) for t in agent.ctx.turns if t.audio is not None):.1f}s of audio history)")
print("\nRecent turns:")
agent.ctx.transcript(n=8)

# agent.reset()   # clear history, keep the voice

## Step 14 — Two-speaker dialogue

CSM's original demo generates a *conversation*, not a monologue: two anchors, two speaker ids, one
shared context. Each line is conditioned on every line before it, from both speakers — which is why
the turns actually respond to each other's energy instead of sounding like two separate TTS runs
stitched together.

In [ ]:
DIALOGUE_SEED_B = 1234   # reroll for a different second voice

torch.manual_seed(DIALOGUE_SEED_B)
anchor_b_text = "Honestly? I wasn't expecting that at all. Tell me more about it."
anchor_b_wav = csm_generate(
    [{"role": str(USER), "content": [{"type": "text", "text": anchor_b_text}]}],
    max_seconds=10.0, temperature=0.85,
)
ANCHOR_B = Turn(USER, anchor_b_text, anchor_b_wav)
print("Second voice:"); play(anchor_b_wav, autoplay=False)

script = [
    (ASSISTANT, "So I finally tried that place you kept talking about."),
    (USER,      "No way. And? Was I right or was I right?"),
    (ASSISTANT, "You were right. I hate that you were right, but you were right."),
    (USER,      "I'm framing this moment. Say it one more time for me."),
    (ASSISTANT, "Not a chance. Let's just say the pasta was worth the wait."),
]

dialogue = Conversation(anchors=[VOICE_ANCHOR, ANCHOR_B])
pieces = []
for spk, line in script:
    who = "A" if spk == ASSISTANT else "B"
    print(f"{who}: {line}")
    wav, turns = speak(line, dialogue, speaker=spk, verbose=False)
    dialogue.turns.extend(turns)
    pieces.append(wav)
    pieces.append(np.zeros(int(0.25 * SAMPLE_RATE), dtype=np.float32))  # beat between turns

full = np.concatenate(pieces)
save_wav(full, "dialogue.wav")
print(f"\n{secs(full):.1f}s of dialogue -> dialogue.wav")
play(full, autoplay=False)

## Step 15 — (Optional) Fine-tuning, and the compute-amortization trick

Anchoring gets you a *consistent* voice. Fine-tuning gets you a *specific* one — that's the gap
between this notebook and Sesame's product, which runs a fine-tuned CSM variant.

This cell demonstrates the paper's training scheme, including the memory trick that makes it
tractable:

> The audio decoder processes an effective batch of `B × S × N` (batch × sequence × 32 codebooks),
> which is enormous. Sesame train it on a random **1/16 subset of frames** while codebook 0 trains
> on **every** frame — and report no perceivable difference in decoder loss.

In 🤗 Transformers that's exactly one argument: **`depth_decoder_labels_ratio=1/16`**. The processor
marks the skipped frames with `-101`, so the backbone still sees them and only the depth decoder
skips.

⚠️ **This is a mechanics demo, not a real training run.** It uses LoRA and a handful of steps on a
tiny dataset so it fits a free T4. A voice you'd actually ship needs hours of clean single-speaker
audio. **Restart the runtime before going back to inference** — the LoRA-wrapped model is not the
one the cells above expect.

In [ ]:
RUN_FINETUNE_DEMO = False   # flip to True to run

if RUN_FINETUNE_DEMO:
    from peft import LoraConfig, get_peft_model
    from datasets import load_dataset, Audio

    # swap in your own dataset: needs `text`, `audio`, `speaker_id` columns at 24 kHz
    ds = load_dataset("hf-internal-testing/dailytalk-dummy", split="train")
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

    conversation = [
        {"role": f"{spk}",
         "content": [{"type": "text", "text": txt},
                     {"type": "audio", "path": aud["array"]}]}
        for txt, aud, spk in zip(ds[:4]["text"], ds[:4]["audio"], ds[:4]["speaker_id"])
    ]

    csm_model.train()
    csm_model.codec_model.eval()          # never train the codec
    for p in csm_model.codec_model.parameters():
        p.requires_grad_(False)

    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    peft_model = get_peft_model(csm_model, lora)
    peft_model.print_trainable_parameters()

    # ---- the compute-amortization knob from the paper -----------------------
    AMORTIZATION_RATIO = 1 / 16   # 1.0 = train the depth decoder on every frame (memory-hungry)

    inputs = csm_processor.apply_chat_template(
        conversation, tokenize=True, return_dict=True,
        output_labels=True,
        depth_decoder_labels_ratio=AMORTIZATION_RATIO,
    ).to(DEVICE)
    if inputs.get("input_values") is not None:
        inputs["input_values"] = inputs["input_values"].to(CSM_DTYPE)

    opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=1e-4)

    for step in range(5):
        out = peft_model(**inputs)
        out.loss.backward()
        opt.step(); opt.zero_grad()
        print(f"step {step}  total {out.loss.item():.4f}  "
              f"backbone {out.backbone_loss.item():.4f}  depth {out.depth_decoder_loss.item():.4f}")

    print("\nMechanics confirmed. peft_model.save_pretrained('csm-lora') to keep the adapter.")
    print("Restart the runtime before returning to the inference cells above.")
else:
    print("Skipped. Set RUN_FINETUNE_DEMO = True to run it.")

## Notes, limits, and where this differs from Sesame

**What's faithful to the paper**

- Interleaved text+audio conditioning — the context buffer keeps waveforms for both speakers
- Speaker-id tokens (`[0]` / `[1]`) so the model tracks whose voice is whose
- The two-transformer split: backbone → codebook 0 (semantic), depth decoder → codebooks 1–31 (acoustic)
- Mimi at 24 kHz / 12.5 Hz, with the frame budget computed against the real 2048-position window
- Compute amortization (`depth_decoder_labels_ratio = 1/16`) in the fine-tuning cell

**What's different, honestly**

| | Sesame's demo | This notebook |
|---|---|---|
| Voice | Fine-tuned CSM variant | Base CSM-1B + a pinned anchor |
| Size | Up to Medium (8B / 300M) | Tiny (1B / 100M) — the only open one |
| Turn-taking | Full duplex, can be interrupted | Half duplex, fixed-length recording |
| Latency | Sub-second, streaming | ~1–3 s to first audio on a T4 |
| Watermark | `silentcipher` on output | **None** — see below |

The duplex gap isn't laziness on our part; it's the open problem Sesame themselves flag as future
work. CSM decides *how* to speak, never *when*. Real turn-taking needs a model of conversational
dynamics that hasn't been released.

**Tuning cheatsheet**

| Symptom | Fix |
|---|---|
| Voice changes between turns | Anchor got evicted or is too short — check `VOICE_ANCHOR.cost()`, reroll `VOICE_SEED` |
| Flat, robotic delivery | Context too thin. Let the conversation run longer; keep audio in the turns |
| Static / garbled audio | You're on fp16 with a T4 — set `CSM_USE_FP16_ON_T4 = False` |
| Rambling, unspeakable replies | Tighten `SYSTEM_PROMPT`; lower `max_new_tokens` |
| Slurring or drift on long sentences | Lower `max_chars` in `split_sentences` |
| OOM | `LLM_MODEL_ID = "google/gemma-3-1b-it"`, or `ASR_MODEL_ID = "openai/whisper-base"` |
| Too slow | `whisper-base`, shorter replies, or a paid L4/A100 runtime (which also unlocks bf16) |

**Responsible use** — Sesame's license prohibits impersonating real individuals without consent,
deception and misinformation, and illegal or harmful use. Two things worth taking seriously:

1. **Watermarking.** Sesame's own repo applies an imperceptible `silentcipher` watermark to every
   output and asks people to keep it. The 🤗 Transformers integration this notebook uses does not
   include it. If you deploy anything built from this, add it back.
2. **Consent.** The "use my own voice" cell is voice cloning. Record yourself, or someone who has
   actually agreed — not a clip off the internet.

**Sources**

- [Crossing the uncanny valley of conversational voice — Sesame](https://www.sesame.com/research/crossing_the_uncanny_valley_of_voice)
- [SesameAILabs/csm on GitHub](https://github.com/SesameAILabs/csm)
- [`sesame/csm-1b` model card](https://huggingface.co/sesame/csm-1b)
- [CSM in 🤗 Transformers](https://huggingface.co/docs/transformers/model_doc/csm)
- [Mimi codec (Kyutai)](https://huggingface.co/kyutai/mimi)
- [Gemma 3](https://huggingface.co/blog/gemma3)